In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:1','cuda:6'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

('cuda:1', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 79361)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:4', 'NVIDIA A100-SXM4-80GB', 62805)

# #### Device() ####
# device = cuda:2

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:2)
# edge_attr                (32464, 16)              Tensor (cuda:2)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:2)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [2]:
brca.x.shape

torch.Size([1172, 4373, 1])

In [3]:
device

'cuda:2'

---

In [4]:
from modules.layers import AttentionSetPooling
from modules.model import LatentClassifier
from modules.norm import ZlogNorm
from modules.train import grid
from functools import partial
import torch.nn as nn

In [5]:
model_template = partial(
    LatentClassifier,
    dataset=dataset,
    
    # layers
    norm_class=ZlogNorm,
    encoder_class=nn.Linear,
    pooling_class=AttentionSetPooling,
    mlp=False,

    # # layer params
    # act_fn=nn.ReLU, 
    # norm_fn='layer', 
    # end_fn=False,
)

model_grid = grid(
    model_template,
    # prefix='hi',
    # suffix='bye',
    embed_dim=[32,64],
    hidden_dims=[2]
)

model_grid

{'embeddim32_hiddendims2': LatentClassifier(
   (encoder): Encoder(
     (norm): ZlogNorm()
     (node_encoder): Sequential(
       (layers): ModuleList(
         (0): Linear(in_features=1, out_features=32, bias=True)
         (1): ReLU()
         (2): Linear(in_features=32, out_features=32, bias=True)
         (3): ReLU()
         (4): Linear(in_features=32, out_features=32, bias=True)
       )
     )
   )
   (latent): Latent(
     (pooling): AttentionSetPooling(
       (lin): Sequential(
         (layers): ModuleList(
           (0): Linear(in_features=32, out_features=32, bias=True)
           (1): ReLU()
           (2): Linear(in_features=32, out_features=32, bias=True)
           (3): ReLU()
           (4): Linear(in_features=32, out_features=1, bias=True)
         )
       )
     )
   )
   (classifier): Sequential(
     (layers): ModuleList(
       (0): Linear(in_features=32, out_features=32, bias=True)
       (1): ReLU()
       (2): Linear(in_features=32, out_features=32, bias=T

---

In [6]:
from modules.train import ClassifTrainer, Experiment

In [7]:
trainer_template = partial(
    ClassifTrainer,
    loss_class=nn.CrossEntropyLoss,
)

trainer_grid = grid(
    trainer_template,
    lr=[1e-1,1e-2,1e-3,1e-4]
)

trainer_grid

{'lr1e-1': <modules.train.ClassifTrainer at 0x7f4ac07c7fb0>,
 'lr1e-2': <modules.train.ClassifTrainer at 0x7f4ac02e32c0>,
 'lr1e-3': <modules.train.ClassifTrainer at 0x7f4ac02e3110>,
 'lr1e-4': <modules.train.ClassifTrainer at 0x7f4ac02e30b0>}

In [8]:
for i in trainer_grid:
    print(trainer_grid[i].lr)

0.1
0.01
0.001
0.0001


---

In [9]:
from modules.train import Loader

loader = Loader(
    dataset=dataset,
    device=device
)

trainer = ClassifTrainer(
    lr=1e-3,
    loss_class=nn.CrossEntropyLoss,
    weight_method='balanced',
    report_metrics=['loss','accuracy']
)

model = LatentClassifier(
    dataset=dataset,
    method='node',
    embed_dim=64,

    # layers
    norm_class=ZlogNorm,
    encoder_class=nn.Linear,
    pooling_class=AttentionSetPooling,
    mlp=False,

    # layer params
    hidden_dims=2,
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,
)

trainer.run(
    model=model,
    loader=loader,
    num_epochs=10,
    verbose=True,
)

  0%|          | 0/10 [00:00<?, ?it/s]

Test	 loss=1.4003    accuracy=0.4457



In [ ]:
expt = Experiment(
    num_trials=1,
    num_epochs=10,
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_config(
    name='test',
    model=model,
    trainer=trainer
)

expt.add_config(
    name='test2',
    model=model,
    trainer=trainer
)

expt.run_experiment('timer_test')

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Trial 1/1, Config: 1/2 (test),	 loss=1.5064    accuracy=0.3829


  0%|          | 0/10 [00:00<?, ?it/s]

Trial 1/1, Config: 2/2 (test2),	 loss=1.5037    accuracy=0.4914


---

In [11]:
expt = Experiment(
    num_trials=1,
    num_epochs=10,
    dataset=dataset,
    device=device,
    batch_size=128
)

model_grid = grid(
    model_template,
    embed_dim=[32,64],
    hidden_dims=[2,3],
    prefix='hello',
    suffix='goodbye'
)

trainer_grid = grid(
    trainer_template,
    lr=[1e-2,1e-3,1e-4],
    prefix='hola',
    suffix='ciao'
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid,
    prefix='hi',
    suffix='bye'
)

In [12]:
model_grid = grid(
    model_template,
    embed_dim=[32,64],
    hidden_dims=[2,3],
    prefix='hello',
    suffix='goodbye'
)

trainer_grid = grid(
    trainer_template,
    lr=[1e-2,1e-3,1e-4],
    prefix='hola',
    suffix='ciao'
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid,
    prefix='hi',
    suffix='bye'
)

In [13]:
for i in expt.configs.keys():
    print(i)

hi_hello_embeddim32_hiddendims2_goodbye_hola_lr1e-2_ciao_bye
hi_hello_embeddim32_hiddendims2_goodbye_hola_lr1e-3_ciao_bye
hi_hello_embeddim32_hiddendims2_goodbye_hola_lr1e-4_ciao_bye
hi_hello_embeddim32_hiddendims3_goodbye_hola_lr1e-2_ciao_bye
hi_hello_embeddim32_hiddendims3_goodbye_hola_lr1e-3_ciao_bye
hi_hello_embeddim32_hiddendims3_goodbye_hola_lr1e-4_ciao_bye
hi_hello_embeddim64_hiddendims2_goodbye_hola_lr1e-2_ciao_bye
hi_hello_embeddim64_hiddendims2_goodbye_hola_lr1e-3_ciao_bye
hi_hello_embeddim64_hiddendims2_goodbye_hola_lr1e-4_ciao_bye
hi_hello_embeddim64_hiddendims3_goodbye_hola_lr1e-2_ciao_bye
hi_hello_embeddim64_hiddendims3_goodbye_hola_lr1e-3_ciao_bye
hi_hello_embeddim64_hiddendims3_goodbye_hola_lr1e-4_ciao_bye
hi_hello_embeddim32_hiddendims2_goodbye_hola_lr1e-2_ciao_bye_0
hi_hello_embeddim32_hiddendims2_goodbye_hola_lr1e-3_ciao_bye_0
hi_hello_embeddim32_hiddendims2_goodbye_hola_lr1e-4_ciao_bye_0
hi_hello_embeddim32_hiddendims3_goodbye_hola_lr1e-2_ciao_bye_0
hi_hello_embeddi